<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mase_EDGE - Optuna Study for NAS and Hyperparameters

Mase EDGE performs Optuna study for a specific Android phone that the user specify.

## 1. Defining the Search Space

First define the search space for Optuna study. Include the NAS search space and metrics like the phone hardware delegation split ratio for the search process.

In [ ]:
import torch.nn as nn
from chop.nn.modules import Identity

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}

## 2. Writing a Model Constructor

We define the following function, which will get called in each iteration of the search process. The function is passed the `trial` argument, which is an Optuna object that comes with many functionalities - see the [Trial documentation](https://optuna.readthedocs.io/en/stable/reference/trial.html) for more details. Here, we use the `trial.suggest_int` and `trial.suggest_categorical` functions to trigger the chosen sampler to choose parameter choices and layer types. The suggested integer is the index into the search space for each parameter, which we defined in the previous cell.

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools.utils import deepsetattr

def make_identity_linear(dim: int) -> nn.Linear:
    """
    MASE-compatible replacement for nn.Identity().
    Acts as an identity transform but remains a 'linear' mase_op.
    """
    layer = nn.Linear(dim, dim, bias=False)
    nn.init.eye_(layer.weight)
    layer.weight.requires_grad_(False)
    return layer


def construct_model(trial):
    # Load base config
    config = AutoConfig.from_pretrained(checkpoint)

    # -----------------------------
    # 1. GLOBAL ARCHITECTURE PARAMS
    # (must all be in GridSampler search_space)
    # -----------------------------
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        value = trial.suggest_categorical(param, search_space[param])
        setattr(config, param, value)

    # -----------------------------
    # 2. VALIDITY CONSTRAINTS
    # -----------------------------
    # BERT requirement: hidden_size divisible by num_heads
    if config.hidden_size % config.num_heads != 0:
        raise optuna.TrialPruned()

    # -----------------------------
    # 3. BUILD MODEL
    # -----------------------------
    trial_model = AutoModelForSequenceClassification.from_config(config)

    # -----------------------------
    # 4. DETERMINISTIC LAYER REWRITE
    # (NO dynamic trial parameters)
    # -----------------------------
    # Example rule:
    # Replace square Linear layers with Identity
    # This keeps the model grid-static
    for name, layer in trial_model.named_modules():
        if (
            isinstance(layer, nn.Linear)
            and layer.in_features == layer.out_features
        ):
            deepsetattr(
                trial_model,
                name,
                make_identity_linear(layer.in_features),
            )

    return trial_model



## 3. Defining the Objective Function

Next, we define the objective function for the search, which gets called on each trial. In each trial, we create a new model instace with chosen hyperparameters according to the defined sampler. We then use the `get_trainer` utility in Mase to run a training loop on the IMDb dataset for a number of epochs. Finally, we use `evaluate` to report back the classification accuracy on the test split.

In [ ]:
from chop.tools import get_trainer


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    # Set the model as an attribute so we can fetch it later
    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]

## 4. Launching the Search

Optuna provides a number of samplers, for example:

* **GridSampler**: iterates through every possible combination of hyperparameters in the search space
* **RandomSampler**: chooses a random combination of hyperparameters in each iteration
* **TPESampler**: uses Tree-structured Parzen Estimator algorithm to choose hyperparameter values.

You can define the chosen sampler by simply importing from `optuna.samplers` as below.

In [ ]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = GridSampler(search_space)

With all the pieces in place, we can launch the search as follows. The number of trials is set to 1 so you can go get a coffee for 10 minutes, then proceed with the tutorial. However, this will essentially be a random model - for better results, set this to 100 and leave it running overnight!

In [ ]:
import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=4,
    timeout=60 * 60 * 24,
)

Fetch the model associated with the best trial as follows, and export to be used in future tutorials. In Tutorial 6, we'll see how to run mixed-precision quantization search on top of the model we've just found through NAS to further find the optimal quantization mapping.

In [ ]:
from pathlib import Path
import dill

# model = study.best_trial.user_attrs["model"].cpu()

# with open(f"{Path.home()}/tutorial_5_best_model.pkl", "wb") as f:
#     dill.dump(model, f)

model = study.best_trial.user_attrs["model"].cpu()
with open(f"/content/tutorial_5_best_model_final.pkl", "wb") as f:
    dill.dump(model, f)

## Deploying the Optimized Model with CompressionPipeline

Now, we can run the CompressionPipeline in Mase to run uniform quantization and pruning over the searched model.

In [ ]:
from chop.pipelines import CompressionPipeline
from chop import MaseGraph

mg = MaseGraph(model)
pipe = CompressionPipeline()

quantization_config = {
    "by": "type",
    "default": {
        "config": {
            "name": None,
        }
    },
    "linear": {
        "config": {
            "name": "integer",
            # data
            "data_in_width": 8,
            "data_in_frac_width": 4,
            # weight
            "weight_width": 8,
            "weight_frac_width": 4,
            # bias
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
}

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg, _ = pipe(
    mg,
    pass_args={
        "quantize_transform_pass": quantization_config,
        "prune_transform_pass": pruning_config,
    },
)

Finally, export the MaseGraph for the compressed checkpoint to be used in future tutorials for hardware generation and distributed deployment.

In [ ]:
mg.export(f"{Path.home()}/tutorial_5_nas_compressed", save_format="state_dict")

In [ ]:
from pathlib import Path

out_dir = Path("/content") / "tutorial_5_nas_compressed"
mg.export(str(out_dir), save_format="state_dict")


In [ ]:
# Code for Lab 2 Task 1
from chop.pipelines import CompressionPipeline
from chop import MaseGraph
import optuna
from optuna.samplers import GridSampler, RandomSampler, TPESampler
import dill
import matplotlib.pyplot as plt

grid_sampler   = GridSampler(search_space)
tpe_sampler    = TPESampler()
random_sampler = RandomSampler()

samplers      = [grid_sampler, tpe_sampler, random_sampler]
sampler_names = ["GridSampler"RandomSampler"]

# For plotting: sampler_name -> list of best accuracies vs max_trials
sampler_curves = {name: [] for name in sampler_names}

# -------------------------
# Main loop: keep your idea
# -------------------------

for s, s_name in zip(samplers, sampler_names):

    # New study for each (sampler, max_trials) pair
    study = optuna.create_study(
        direction="maximize",
        study_name=f"bert-tiny-nas-{s_name}",
        sampler=s,
    )

    study.optimize(
        objective,
        n_trials=25,        # run EXACTLY max_trials trials
        timeout=60 * 60 * 24,
    )

    # Best accuracy after 'max_trials' trials
    best_acc = study.best_value
    sampler_curves[s_name].append(best_acc)

    best_trial = study.best_trial
    best_params = study.best_params
    values = [t.value for t in study.trials]

    # Optional: save current best model
    model = study.best_trial.user_attrs["model"].cpu()
    with open(f"/content/tutorial_5_best_model_{s_name}.pkl", "wb") as f:
        dill.dump(model, f)

        # Optional: compression pipeline (unchanged)
# mg = MaseGraph(model)
# pipe = CompressionPipeline()

# quantization_config = {
#     "by": "type",
#     "default": {"config": {"name": None}},
#     "linear": {
#         "config": {
#             "name": "integer",
#             "data_in_width": 8,
#             "data_in_frac_width": 4,
#             "weight_width": 8,
#             "weight_frac_width": 4,
#             "bias_width": 8,
#             "bias_frac_width": 4,
#         }
#     },
# }

# pruning_config = {
#     "weight": {
#         "sparsity": 0.5,
#         "method": "l1-norm",
#         "scope": "local",
#     },
#     "activation": {
#         "sparsity": 0.5,
#         "method": "l1-norm",
#         "scope": "local",
#     },
# }

# mg, _ = pipe(
#     mg,
#     pass_args={
#         "quantize_transform_pass": quantization_config,
#         "prune_transform_pass": pruning_config,
#     },
# )

# # -------------------------
# # Plot: max_trials vs best accuracy
# # -------------------------

plt.figure(figsize=(10, 6))

for s_name in sampler_names:
    trials = list(range(1, len(sampler_curves[s_name]) + 1))
    plt.plot(trials, sampler_curves[s_name],
             marker='o', label=s_name, linewidth=2, markersize=4)

plt.xlabel('Number of Trials')
plt.ylabel('Best Accuracy')
plt.title('Sampler Comparison: Best Accuracy vs Trials')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- 1. Enter your data ----
trials = np.arange(1, 26)

grid = np.array([
    0.8354, 0.7958, 0.8248, 0.8456, 0.8258,
    0.8159, 0.8354, 0.6501, 0.8354, 0.80596,
    0.8352, 0.79848, 0.80832, 0.82316, 0.80384,
    0.8456, 0.68224, 0.81228, 0.80832, 0.82888,
    0.65016, 0.68224, 0.83232, 0.82528, 0.8456
])

tpe = np.array([
    0.82528, 0.82528, 0.81484, 0.81592, 0.68224,
    0.81484, 0.81116, 0.82692, 0.81304, 0.81592,
    0.82692, 0.82692, 0.82316, 0.82692, 0.82692,
    0.82692, 0.79848, 0.83048, 0.83048, 0.83048,
    0.80384, 0.83048, 0.83048, 0.83048, 0.81304
])

rand = np.array([
    0.81736, 0.82528, 0.82528, 0.80384, 0.83232,
    0.83232, 0.80832, 0.80512, 0.80764, 0.82484,
    0.80764, 0.8184, 0.8352, 0.82528, 0.80832,
    0.83232, 0.82692, 0.80512, 0.80764, 0.80512,
    0.80384, 0.82888, 0.83048, 0.68224, 0.8456
])

# ---- 2. Cumulative best curves ----
grid_best = np.maximum.accumulate(grid)
tpe_best = np.maximum.accumulate(tpe)
rand_best = np.maximum.accumulate(rand)

# ---- 3. Plot: dots + best-accuracy lines ----
plt.figure(figsize=(10, 6))

# Scatter: raw trial accuracies
plt.scatter(trials, grid,  color='tab:blue',  s=25, label='Grid (trials)')
plt.scatter(trials, tpe,   color='tab:orange', s=25, label='TPE (trials)')
plt.scatter(trials, rand,  color='tab:green', s=25, label='Random (trials)')

# Lines: cumulative best accuracy
plt.plot(trials, grid_best, color='tab:blue',  linewidth=2, linestyle='-',  label='Grid (best so far)')
plt.plot(trials, tpe_best,  color='tab:orange',linewidth=2, linestyle='-',  label='TPE (best so far)')
plt.plot(trials, rand_best, color='tab:green', linewidth=2, linestyle='-',  label='Random (best so far)')

plt.xlabel('Trial')
plt.ylabel('Evaluation Accuracy')
plt.title('Sampler Comparison: Accuracy per Trial and Best-So-Far')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
